# LinguaForge — Listen pillar (audio → learning card)

Pipes a 43-second **Cherokee Morning Song** (Wikimedia Commons, CC-BY-SA
4.0, uploaded by user Bono.Ruma, 22 June 2024) through:

1. Base **`google/gemma-4-E4B-it`** multimodal *audio* understanding.
2. Same model + the **LinguaForge LoRA** for endangered languages.

and compares the structured learning-card output that comes back. Saves
`listen_results.json` plus the raw transcripts.

In [ ]:
import os
if not os.environ.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
assert os.environ.get('HF_TOKEN'), 'HF_TOKEN missing.'
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print('HF token:', os.environ['HF_TOKEN'][:8], '...', os.environ['HF_TOKEN'][-4:])

In [ ]:
!pip install -qU unsloth==2026.5.2 peft transformers accelerate sentencepiece hf_transfer soundfile
!apt-get install -qq -y ffmpeg > /tmp/apt.log 2>&1 || true

## 1. Download the Cherokee Morning Song

In [ ]:
import os, urllib.request
AUDIO_URL = 'https://upload.wikimedia.org/wikipedia/commons/5/51/Morning-song-on-Cherokee_%281%29.opus'
os.makedirs('/kaggle/working/audio', exist_ok=True)
OPUS_PATH = '/kaggle/working/audio/cherokee_morning_song.opus'
WAV_PATH  = '/kaggle/working/audio/cherokee_morning_song.wav'
req = urllib.request.Request(AUDIO_URL, headers={'User-Agent': 'LinguaForge/0.1 (hackathon; zcgf111@163.com)'})
with urllib.request.urlopen(req, timeout=30) as r, open(OPUS_PATH, 'wb') as f:
    f.write(r.read())
print('Downloaded:', os.path.getsize(OPUS_PATH), 'bytes')
!ffmpeg -y -i {OPUS_PATH} -ar 16000 -ac 1 {WAV_PATH} 2>/dev/null
print('Resampled to 16k mono WAV:', os.path.getsize(WAV_PATH), 'bytes')

## 2. Load Gemma 4 E4B (multimodal) + LinguaForge LoRA

In [ ]:
import glob
candidates = glob.glob('/kaggle/input/**/adapter_model.safetensors', recursive=True)
assert candidates, 'Adapter dataset not mounted.'
ADAPTER_DIR = os.path.dirname(candidates[0])
print('ADAPTER_DIR =', ADAPTER_DIR)

from unsloth import FastLanguageModel

# IMPORTANT: do NOT call unsloth's get_chat_template here — it overwrites the
# multimodal Gemma 4 chat template with a text-only one, which breaks audio
# placeholder insertion. The processor's default template already supports
# multimodal content lists with {'type': 'audio'} entries.
model, tok = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_DIR,
    max_seq_length=2048,
    load_in_4bit=True,
    token=os.environ['HF_TOKEN'],
)
FastLanguageModel.for_inference(model)
print('processor type:', type(tok).__name__)
print('model class    :', type(model).__name__)
print('has chat template (first 120 chars):', (getattr(tok, 'chat_template', None) or '')[:120])

## 3. Run multimodal inference (audio in, learning card out)

Gemma 4 E4B is multimodal: the processor accepts an `audio=` kwarg alongside
`text=`. We toggle the LoRA adapter on/off to compare base vs LinguaForge.

In [ ]:
import contextlib, soundfile as sf, torch
audio_np, sr = sf.read(WAV_PATH)
print('audio loaded:', audio_np.shape, 'sr =', sr, 'duration =', round(len(audio_np)/sr, 2), 's')

@contextlib.contextmanager
def use_adapter(enabled):
    if enabled:
        yield; return
    if hasattr(model, 'disable_adapter'):
        with model.disable_adapter():
            yield; return
    try:
        model.disable_adapter_layers(); yield
    finally:
        model.enable_adapter_layers()

PROMPT = (
    "You are LinguaForge, a multilingual tutor for endangered languages.\n"
    "The user has given you a 43-second audio recording. Please:\n"
    "1. Identify the language and musical style.\n"
    "2. Describe the vocal characteristics (intonation, rhythm, vowel inventory).\n"
    "3. Suggest THREE Cherokee vocabulary cards a learner could derive from this clip, "
    "each with: Cherokee (syllabary) | romanization | English gloss | cultural note. "
    "If you cannot transcribe the audio reliably, say so honestly and give cards based on "
    "the *known* tradition of Cherokee morning songs instead."
)

def listen(with_lora):
    msgs = [
        {'role': 'user', 'content': [
            {'type': 'audio', 'audio': audio_np},
            {'type': 'text', 'text': PROMPT},
        ]},
    ]
    inputs = tok.apply_chat_template(
        msgs,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
    ).to(model.device)
    n_audio_tokens = int((inputs['input_ids'] == tok.tokenizer.convert_tokens_to_ids('<audio_soft_token>')).sum().item()) if hasattr(tok, 'tokenizer') else -1
    print(f'input tokens: {inputs["input_ids"].shape}, audio_soft_tokens detected: {n_audio_tokens}')
    with use_adapter(with_lora), torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            use_cache=True,
        )
    text_tok = getattr(tok, 'tokenizer', tok)
    gen = text_tok.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    return gen.strip()

print('\n=== Base Gemma 4 E4B (no LoRA) ===\n')
base_out = listen(with_lora=False)
print(base_out)
print('\n=== + LinguaForge LoRA ===\n')
lora_out = listen(with_lora=True)
print(lora_out)

In [ ]:
import json
results = {
    'audio_source': {
        'title': 'Morning-song-on-Cherokee (1).opus',
        'url': AUDIO_URL,
        'license': 'CC-BY-SA 4.0',
        'author': 'Bono.Ruma (Wikimedia Commons)',
        'duration_sec': round(len(audio_np)/sr, 2),
    },
    'prompt': PROMPT,
    'base_output': base_out,
    'lora_output': lora_out,
}
with open('/kaggle/working/listen_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print('saved /kaggle/working/listen_results.json')
!ls -la /kaggle/working